# word2vec (Skip-gram + Negative Sampling) — Paper-to-Code Mock (Colab)

**Paper:** Distributed Representations of Words and Phrases and their Compositionality (Mikolov et al., 2013) — https://arxiv.org/abs/1310.4546

Medium mock (~60 min). Fill in the `SGNS` stub, run the synthetic-corpus toy task, then the sanity checks. Reference solution in the last cell.

Key trick: a full softmax over the vocab is O(V); **negative sampling** turns it into O(k) binary logistic classification — true (center, context) pair vs `k` random negatives.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement Skip-gram with Negative Sampling (YOUR TASK)

Two embedding tables: `center` (v) and `context` (u). For center `c`, true context `o`, and `k` negatives `n`, the loss is

`-log σ(u_o·v_c) - Σ_n log σ(-u_n·v_c)`  (mean over the batch).

Also fill in `draw_negatives`, which samples `(num, k)` negatives from a given distribution.

In [ ]:
class SGNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)    # v: word as center/input
        self.context = nn.Embedding(vocab_size, dim)   # u: word as predicted neighbour
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.context.weight, -0.5 / dim, 0.5 / dim)

    def loss(self, center_ids, context_ids, neg_ids):
        # center_ids:(B,)  context_ids:(B,)  neg_ids:(B,k)
        # TODO: v_c = self.center(center_ids)            # (B, d)
        # TODO: u_o = self.context(context_ids)          # (B, d)
        # TODO: u_n = self.context(neg_ids)              # (B, k, d)
        # TODO: pos = F.logsigmoid((u_o * v_c).sum(-1))                       # pull together
        # TODO: neg_logit = torch.bmm(u_n, v_c.unsqueeze(-1)).squeeze(-1)     # (B, k)
        # TODO: neg = F.logsigmoid(-neg_logit)                               # push apart (minus!)
        # TODO: return -(pos + neg.sum(-1)).mean()       # scalar
        raise NotImplementedError("fill me in")


def draw_negatives(num, k, sampling_weights, generator=None):
    # TODO: sample num*k ids with replacement from sampling_weights, reshape to (num, k)
    raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (synthetic-corpus toy task)
4 topics x 5 words. Each sentence emits one topic's words, so words co-occur only within their topic. Train SGNS with NO labels (just co-occurrence) and check same-topic cosine ≫ cross-topic cosine.

In [ ]:
def build_corpus(seed=0):
    g = torch.Generator().manual_seed(seed)
    topics = [list(range(t * 5, t * 5 + 5)) for t in range(4)]   # 4 topics x 5 words = vocab 20
    vocab_size = 20
    sentences = []
    for _ in range(800):
        t = torch.randint(0, len(topics), (1,), generator=g).item()
        words = topics[t][:]
        perm = torch.randperm(len(words), generator=g).tolist()
        sentences.append([words[i] for i in perm])
    window, centers, contexts = 2, [], []
    for s in sentences:
        for i, c in enumerate(s):
            for j in range(max(0, i - window), min(len(s), i + window + 1)):
                if j != i:
                    centers.append(c); contexts.append(s[j])
    centers, contexts = torch.tensor(centers), torch.tensor(contexts)
    weights = torch.bincount(centers, minlength=vocab_size).float().pow(0.75)  # unigram^0.75
    return centers, contexts, weights / weights.sum(), topics, vocab_size


def train_sgns(seed=0, dim=16, k=5, epochs=60, lr=0.05, batch=256):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    centers, contexts, weights, topics, V = build_corpus(seed)
    model = SGNS(V, dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n, losses = centers.shape[0], []
    for _ in range(epochs):
        perm = torch.randperm(n, generator=g)
        ep, nb = 0.0, 0
        for start in range(0, n, batch):
            idx = perm[start:start + batch]
            neg = draw_negatives(idx.shape[0], k, weights, generator=g)
            loss = model.loss(centers[idx], contexts[idx], neg)
            opt.zero_grad(); loss.backward(); opt.step()
            ep += loss.item(); nb += 1
        losses.append(ep / nb)
    return model, topics, losses


def group_similarities(emb, topics):
    sim = F.normalize(emb, dim=-1) @ F.normalize(emb, dim=-1).t()
    wt = {w: ti for ti, ws in enumerate(topics) for w in ws}
    same, diff = [], []
    for a in range(emb.shape[0]):
        for b in range(a + 1, emb.shape[0]):
            (same if wt[a] == wt[b] else diff).append(sim[a, b])
    return torch.stack(same).mean().item(), torch.stack(diff).mean().item()


model, topics, losses = train_sgns(seed=0)
same, diff = group_similarities(model.center.weight.detach(), topics)
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"same-topic cos = {same:+.3f}   diff-topic cos = {diff:+.3f}")

# nearest neighbour of word 0 (topic 0 = words 0-4)
emb = model.center.weight.detach()
sim0 = (F.normalize(emb, dim=-1) @ F.normalize(emb, dim=-1).t())[0].clone(); sim0[0] = -2.0
print("nearest neighbour of word 0:", int(sim0.argmax().item()))

## 3. Sanity checks

In [ ]:
topics = [list(range(t * 5, t * 5 + 5)) for t in range(4)]

# 1) loss is a scalar
m = SGNS(20, 16)
c, o, neg = torch.tensor([0, 1, 2]), torch.tensor([1, 2, 3]), torch.tensor([[4, 5], [6, 7], [8, 9]])
L = m.loss(c, o, neg)
assert L.dim() == 0

# 2) at init no structure (same ~ diff ~ 0)
s0, d0 = group_similarities(SGNS(20, 16).center.weight.detach(), topics)
assert abs(s0) < 0.2 and abs(d0) < 0.2

# 3) negative sampling draws the requested count from the vocab
neg = draw_negatives(7, 5, torch.ones(20) / 20)
assert neg.shape == (7, 5) and int(neg.min()) >= 0 and int(neg.max()) < 20

# 4) loss decreases
_, _, losses = train_sgns(seed=0)
assert losses[-1] < losses[0]

# 5) after training same-topic cos > diff-topic cos (the demo)
model, topics, _ = train_sgns(seed=0)
same, diff = group_similarities(model.center.weight.detach(), topics)
assert same > diff + 0.2

# 6) gradient flows to BOTH tables
m = SGNS(20, 16)
m.loss(torch.tensor([0, 1]), torch.tensor([1, 2]), torch.tensor([[3, 4], [5, 6]])).backward()
assert m.center.weight.grad.abs().sum() > 0 and m.context.weight.grad.abs().sum() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class SGNSSolution(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)
        self.context = nn.Embedding(vocab_size, dim)
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.context.weight, -0.5 / dim, 0.5 / dim)

    def loss(self, center_ids, context_ids, neg_ids):
        v_c = self.center(center_ids)               # (B, d)
        u_o = self.context(context_ids)             # (B, d)
        u_n = self.context(neg_ids)                 # (B, k, d)
        pos = F.logsigmoid((u_o * v_c).sum(-1))
        neg = F.logsigmoid(-torch.bmm(u_n, v_c.unsqueeze(-1)).squeeze(-1))
        return -(pos + neg.sum(-1)).mean()


def draw_negatives_solution(num, k, sampling_weights, generator=None):
    flat = torch.multinomial(sampling_weights, num * k, replacement=True, generator=generator)
    return flat.view(num, k)